advanced agentic rag (filesystem based)

In [ ]:
import os
import re
from typing import Any, Dict

from rich import print as rptint
from dotenv import load_dotenv
import numpy as np
from pymongo import MongoClient

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers.hybrid_search import MongoDBAtlasHybridSearchRetriever

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI')
DB_NAME = 'agent_db'
COLLECTION_NAME = 'cxi-travel-ins-collection'
ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index"

client = MongoClient(MONGODB_URI)
MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

In [ ]:
client = MongoClient(MONGODB_URI)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)
vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embeddings,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
    embedding_key="embedding",
    text_key="text"
)
retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store,
    search_index_name="search_index",
    top_k=5,
    fulltext_penalty=50,
    vector_penalty=50,
    post_filter=[
        {
            '$project': {
                'embedding': 0,
            }
        }
    ]
)